# Task 03 – Advanced Statistical Analysis & Hypothesis Testing

This notebook demonstrates the statistical procedures required for Task 03:
- Shapiro–Wilk and Kolmogorov–Smirnov normality tests
- Independent two-sample t-test and Mann–Whitney U test
- One-way and two-way ANOVA
- Tukey HSD post-hoc comparisons
- Null/alternative hypotheses and a confidence interval

**Dataset note:** The notebook uses the built-in Wine Recognition scientific dataset from scikit-learn so it can run without downloading a separate file. If the Academy specifies its own dataset, use that dataset and update the column selections and group definitions in Section 2 before submission.


## 1. Install and import libraries

In [ ]:
# If any package is missing, uncomment and run:
# %pip install pandas numpy scipy statsmodels matplotlib seaborn scikit-learn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.datasets import load_wine
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

sns.set_theme(style="whitegrid")
ALPHA = 0.05


## 2. Load and inspect the dataset

In [ ]:
# Built-in scientific dataset. Replace this section with pd.read_csv("your_file.csv")
# if your Academy task specifies a particular dataset.
wine = load_wine(as_frame=True)
df = wine.frame.copy()
df = df.rename(columns={"target": "wine_class"})
df["wine_class"] = df["wine_class"].astype("category")

print("Dataset shape:", df.shape)
display(df.head())
print("\\nColumns:", list(df.columns))
print("\\nClass counts:")
display(df["wine_class"].value_counts().sort_index())


## 3. Define the research question and hypotheses

**Research question:** Does mean color intensity differ among the three wine classes, and does the difference vary across alcohol-level groups?

- One-way ANOVA null hypothesis (H₀): all wine-class mean color-intensity values are equal.
- One-way ANOVA alternative hypothesis (H₁): at least one wine-class mean differs.
- Two-sample tests compare wine class 0 with wine class 1.
- Two-way ANOVA evaluates the effects of wine class, alcohol group, and their interaction on color intensity.

The significance level is α = 0.05.


## 4. Exploratory analysis and distribution plots

In [ ]:
print(df[["alcohol", "color_intensity", "malic_acid"]].describe())

plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="color_intensity", hue="wine_class", kde=True, element="step")
plt.title("Color Intensity Distribution by Wine Class")
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="wine_class", y="color_intensity")
plt.title("Color Intensity by Wine Class")
plt.xlabel("Wine class")
plt.ylabel("Color intensity")
plt.show()


## 5. Normality tests: Shapiro–Wilk and Kolmogorov–Smirnov
H₀: the sample is drawn from a normal distribution. H₁: the sample is not normally distributed. The KS test below compares standardized observations with the standard normal distribution.

In [ ]:
normality_rows = []

for cls, group in df.groupby("wine_class", observed=True):
    values = group["color_intensity"].dropna().to_numpy()
    # Shapiro-Wilk is appropriate for these sample sizes
    shapiro_stat, shapiro_p = stats.shapiro(values)

    # Standardize sample before comparing with the standard normal distribution
    z_values = (values - values.mean()) / values.std(ddof=1)
    ks_stat, ks_p = stats.kstest(z_values, "norm")

    normality_rows.append({
        "Wine class": int(cls),
        "Shapiro statistic": shapiro_stat,
        "Shapiro p-value": shapiro_p,
        "KS statistic": ks_stat,
        "KS p-value": ks_p
    })

normality_results = pd.DataFrame(normality_rows)
display(normality_results)

print("Interpretation: for each test, p < 0.05 suggests rejecting the normality null hypothesis.")


## 6. Independent two-sample t-test and Mann–Whitney U test
Compare color intensity for wine class 0 versus wine class 1. For both tests, H₀: the groups do not differ (for the t-test, equal population means; for Mann–Whitney, equal distributions). H₁: there is a difference.

In [ ]:
group0 = df.loc[df["wine_class"] == 0, "color_intensity"].dropna()
group1 = df.loc[df["wine_class"] == 1, "color_intensity"].dropna()

# Welch's independent t-test does not assume equal group variances
t_stat, t_p = stats.ttest_ind(group0, group1, equal_var=False)

# Non-parametric independent-samples test
u_stat, u_p = stats.mannwhitneyu(group0, group1, alternative="two-sided")

print(f"Welch two-sample t-test: statistic={t_stat:.4f}, p-value={t_p:.6g}")
print(f"Mann–Whitney U test: statistic={u_stat:.4f}, p-value={u_p:.6g}")

# 95% Welch confidence interval for difference in means (class 0 minus class 1)
n0, n1 = len(group0), len(group1)
mean_diff = group0.mean() - group1.mean()
se_diff = np.sqrt(group0.var(ddof=1)/n0 + group1.var(ddof=1)/n1)
df_welch = (group0.var(ddof=1)/n0 + group1.var(ddof=1)/n1)**2 / (
    (group0.var(ddof=1)/n0)**2/(n0-1) + (group1.var(ddof=1)/n1)**2/(n1-1)
)
critical_t = stats.t.ppf(0.975, df_welch)
ci_low, ci_high = mean_diff - critical_t*se_diff, mean_diff + critical_t*se_diff
print(f"Mean difference (class 0 - class 1): {mean_diff:.4f}")
print(f"95% confidence interval: [{ci_low:.4f}, {ci_high:.4f}]")


## 7. One-way ANOVA
H₀: all three wine-class mean color-intensity values are equal. H₁: at least one class mean differs.

In [ ]:
class_groups = [
    df.loc[df["wine_class"] == cls, "color_intensity"].dropna().to_numpy()
    for cls in sorted(df["wine_class"].unique())
]
f_stat, anova_p = stats.f_oneway(*class_groups)
print(f"One-way ANOVA: F-statistic={f_stat:.4f}, p-value={anova_p:.6g}")

one_way_model = ols("color_intensity ~ C(wine_class)", data=df).fit()
one_way_table = sm.stats.anova_lm(one_way_model, typ=2)
display(one_way_table)

if anova_p < ALPHA:
    print("Decision: reject H0. There is evidence that at least one class mean differs.")
else:
    print("Decision: fail to reject H0. There is insufficient evidence of a class mean difference.")


## 8. Tukey HSD post-hoc analysis
Run Tukey's test to identify which pairs of wine classes differ. Interpret adjusted p-values using α = 0.05.

In [ ]:
tukey = pairwise_tukeyhsd(
    endog=df["color_intensity"],
    groups=df["wine_class"].astype(str),
    alpha=ALPHA
)
print(tukey)


## 9. Two-way ANOVA with interaction
Create two alcohol groups using the sample median, then test wine class, alcohol group, and their interaction. H₀ for each term: that term has no effect on mean color intensity. H₁: that term has an effect. This is an illustrative factorization; use factors specified by your Academy dataset if different.

In [ ]:
median_alcohol = df["alcohol"].median()
df["alcohol_group"] = np.where(df["alcohol"] >= median_alcohol, "High alcohol", "Low alcohol")

two_way_model = ols(
    "color_intensity ~ C(wine_class) * C(alcohol_group)",
    data=df
).fit()
two_way_table = sm.stats.anova_lm(two_way_model, typ=2)
display(two_way_table)

print("Two-way ANOVA tests the main effects of wine class and alcohol group,")
print("as well as whether their interaction is associated with color intensity.")


## 10. Summary of findings

In [ ]:
summary = pd.DataFrame([
    {"Analysis": "Shapiro-Wilk (class-wise)", "p-value range": f"{normality_results['Shapiro p-value'].min():.4g} to {normality_results['Shapiro p-value'].max():.4g}"},
    {"Analysis": "Kolmogorov-Smirnov (class-wise)", "p-value range": f"{normality_results['KS p-value'].min():.4g} to {normality_results['KS p-value'].max():.4g}"},
    {"Analysis": "Welch two-sample t-test (class 0 vs 1)", "p-value range": f"{t_p:.4g}"},
    {"Analysis": "Mann–Whitney U (class 0 vs 1)", "p-value range": f"{u_p:.4g}"},
    {"Analysis": "One-way ANOVA (three classes)", "p-value range": f"{anova_p:.4g}"}
])
display(summary)

print("Use p-values together with effect sizes, confidence intervals, study design, and diagnostic plots.")
print("Statistical significance does not by itself establish practical importance or causation.")


## 11. Conclusion

This notebook demonstrates normality testing, independent two-sample parametric and non-parametric tests, one-way ANOVA, Tukey HSD post-hoc comparisons, a two-way ANOVA with interaction, and a 95% confidence interval.

**Before submission:** If RobTech Academy supplied a required dataset, replace the built-in Wine dataset with that CSV, update the research question and variable selections, run every cell, review the outputs, and save the notebook with its results. The built-in dataset is a runnable demonstration only and may not satisfy a requirement to use a specific Academy dataset.
